# 01 — PhysicalAI-AV Dataset Exploration

This notebook explores the NVIDIA PhysicalAI-AV dataset as used in SE-CVLA training.

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf
from transformers import AutoTokenizer
from data.loaders.physicalai_av_dataset import build_dataloader

cfg = OmegaConf.load('../configs/base.yaml')
tokenizer = AutoTokenizer.from_pretrained(cfg.model.encoder.language_backbone, trust_remote_code=True)

In [ ]:
# Load one batch
loader = build_dataloader(cfg.data, split='val', tokenizer=tokenizer)
batch = next(iter(loader))

print('images shape:       ', batch.images.shape)
print('ego_state shape:    ', batch.ego_state.shape)
print('agents shape:       ', batch.agents.shape)
print('trajectory_gt shape:', batch.trajectory_gt.shape)

In [ ]:
# Visualise multi-camera images from first sample
cam_names = ['front', 'front_left', 'front_right', 'rear', 'rear_left', 'rear_right']
imgs = batch.images[0]  # (6, C, H, W)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

for ax, img, name in zip(axes.flatten(), imgs, cam_names):
    img_np = img.permute(1, 2, 0).numpy() * std + mean
    ax.imshow(img_np.clip(0, 1))
    ax.set_title(name); ax.axis('off')

plt.suptitle('Multi-Camera Input — Sample 0', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Visualise ground-truth trajectory
traj = batch.trajectory_gt[0].numpy()  # (H, 2)

plt.figure(figsize=(6, 8))
plt.plot(traj[:, 0], traj[:, 1], 'b-o', markersize=3, label='GT trajectory')
plt.plot(0, 0, 'r*', markersize=12, label='Ego at t=0')
plt.xlabel('x (m)'); plt.ylabel('y (m)')
plt.title('Ground-Truth Future Trajectory (6.4s @ 10Hz)')
plt.legend(); plt.axis('equal'); plt.grid(True); plt.show()

In [ ]:
# Agent distribution
num_agents_per_sample = batch.agent_mask.sum(dim=-1).float()
print(f'Agents per sample: mean={num_agents_per_sample.mean():.1f}, '
      f'max={num_agents_per_sample.max():.0f}, min={num_agents_per_sample.min():.0f}')

plt.figure(figsize=(6, 4))
plt.hist(num_agents_per_sample.numpy(), bins=20, edgecolor='black')
plt.xlabel('Number of agents'); plt.ylabel('Count')
plt.title('Agent Count Distribution'); plt.show()